### Exercise 2

Moving the convolution window forward by 2 time steps each time, which is equivalent to downsampling the signal by a factor of 2.

In audio terms:
* You are effectively throwing away high-frequency information
* If done naively, this can cause aliasing

### Exercise 3

In [1]:
import torch
import torch.nn.functional as F

x = torch.tensor([1., 2., 3., 4.])          # shape: (4,)
x = x.unsqueeze(0).unsqueeze(0)             # shape: (1, 1, 4)

y = F.pad(x, pad=(2, 2), mode='reflect')
print(y)
print(y.shape)

tensor([[[3., 2., 1., 2., 3., 4., 3., 2.]]])
torch.Size([1, 1, 8])


In [2]:
X = torch.tensor([[1., 2., 3.],
                  [4., 5., 6.],
                  [7., 8., 9.]])

X = X.unsqueeze(0).unsqueeze(0)   # shape: (1, 1, 3, 3)

Y = F.pad(X, pad=(1, 1, 1, 1), mode='reflect')
print(Y[0, 0])

tensor([[5., 4., 5., 6., 5.],
        [2., 1., 2., 3., 2.],
        [5., 4., 5., 6., 5.],
        [8., 7., 8., 9., 8.],
        [5., 4., 5., 6., 5.]])


### Exercise 4

#### 🔍 1. What does stride actually do computationally?

From the lecture note:

* Stride = how far the kernel moves each step 
* Stride = 1 → compute output at **every location**
* Stride = 2 → compute output at **every other location**

So:

> Larger stride ⇒ **fewer output positions**

---

#### 🧮 2. Count the number of computations

Let’s do a concrete example.

###### Input:

* Image: $ n \times n $
* Kernel: $ k \times k $

---

##### Case A: stride = 1

Output size:
$$
(n - k + 1)^2 \approx n^2
$$

Each output requires:

* $ k^2 $ multiplications

Total compute:
$$
\approx n^2 \cdot k^2
$$

---

##### Case B: stride = 2

Output size:
$$
\left(\frac{n}{2}\right)^2 = \frac{n^2}{4}
$$

Total compute:
$$
\approx \frac{n^2}{4} \cdot k^2
$$

---

##### 🎯 Key result

> **Stride = 2 reduces computation by ~4×**

More generally:

> **Stride = s reduces computation by ~( s^2 )** (in 2D)

---

#### ⚡ 3. Main computational benefits

##### ✅ (1) Fewer convolution operations

You simply evaluate the kernel fewer times.

👉 Biggest direct saving:

* Less multiply-add operations
* Faster forward pass

---

##### ✅ (2) Smaller output tensors

Stride reduces spatial resolution:

* Less memory needed
* Less data passed to next layer

---

##### ✅ (3) Cheaper downstream layers

This is actually huge.

Suppose next layer is another convolution:

* Its cost depends on input size
* Smaller input ⇒ much cheaper computation

👉 Stride compounds savings across layers

---

##### ✅ (4) Acts as built-in downsampling

Instead of:

* convolution + pooling

you can do:

* **strided convolution**

👉 Fewer layers, fewer operations

### Exercise 5

#### 🔍 1. What does stride > 1 do statistically?

From before:

> Stride > 1 = you **sample fewer locations**

So instead of using **all local patches**, you use a **subset of them**.

This has important statistical consequences.

---

#### 🧠 2. Main idea: implicit regularization

Using stride > 1 acts like a form of:

> **regularization via subsampling**

You are forcing the model to rely on **coarser, more robust features**, instead of memorizing fine-grained details.

---

#### 📉 3. Benefit #1: reduces overfitting

With stride = 1:

* You evaluate every possible local patch
* The model can memorize very fine spatial patterns

With stride > 1:

* You skip many locations
* The model sees fewer training “examples” (patches)

👉 This reduces effective model capacity

So:

> Less chance to overfit noise or tiny patterns

---

#### 🌍 4. Benefit #2: encourages invariance

Stride introduces a kind of **translation invariance**.

Why?

Because:

* You don’t care about exact pixel alignment anymore
* Small shifts in the input may not change the output much

👉 The model learns:

> “roughly where things are” instead of “exact pixel position”

This is very useful for:

* images (objects can shift)
* audio (timing can vary slightly)

---

#### 🔊 5. Benefit #3: noise reduction

High-frequency noise tends to vary quickly across nearby positions.

Stride > 1:

* skips some of these fluctuations
* effectively smooths the representation

👉 Similar to:

* downsampling after smoothing in signal processing

So:

> The model focuses more on **stable patterns** than noisy details

### Exercise 6

A stride of 1/2 corresponds to upsampling the input, i.e., increasing the spatial resolution of the output. It can be implemented by inserting zeros between input elements and applying a standard convolution, or more commonly via transposed convolution layers. This is useful in tasks that require reconstructing high-resolution outputs from low-resolution representations, such as image generation, semantic segmentation, and autoencoders.

In [12]:
x = torch.tensor([[1., 2.],
                  [3., 4.]])

x = x.unsqueeze(0).unsqueeze(0)  # shape (1,1,2,2)

In [13]:
kernel = torch.tensor([[1., 1., 1.],
                       [1., 1., 1.],
                       [1., 1., 1.]])

In [14]:
import torch.nn as nn

deconv = nn.ConvTranspose2d(1, 1, kernel_size=3, stride=2, bias=False)

# Manually set weights
deconv.weight.data[0,0] = kernel

y1 = deconv(x)
print("ConvTranspose2d output:\n", y1[0,0])

ConvTranspose2d output:
 tensor([[ 1.,  1.,  3.,  2.,  2.],
        [ 1.,  1.,  3.,  2.,  2.],
        [ 4.,  4., 10.,  6.,  6.],
        [ 3.,  3.,  7.,  4.,  4.],
        [ 3.,  3.,  7.,  4.,  4.]], grad_fn=<SelectBackward0>)


#### The real mechanism

There are two equivalent ways to understand transposed convolution.

##### View A: “insert zeros, then convolve”

Because stride = 2, you can imagine first expanding the 2×2 input by inserting zeros between entries:

Original input:

$$
\begin{bmatrix}
1&2\\
3&4
\end{bmatrix}
$$

After zero insertion:

$$
\begin{bmatrix}
1&0&2\\
0&0&0\\
3&0&4
\end{bmatrix}
$$

Then you apply a normal convolution-like spreading operation with the 3×3 kernel.

Since the kernel is all ones, each nonzero input value spreads to a 3×3 neighborhood.

---

##### View B: “each input value stamps a patch onto the output”

This is usually the most intuitive view.

Each input number creates a 3×3 block in the output, scaled by that number, and these blocks are placed 2 steps apart because the stride is 2.

###### Input entry 1 at top-left

The value 1 creates:

$$
1\cdot K=
\begin{bmatrix}
1&1&1\\
1&1&1\\
1&1&1
\end{bmatrix}
$$

placed at output position (0,0).

Contribution:

$$
\begin{bmatrix}
1&1&1&0&0\\
1&1&1&0&0\\
1&1&1&0&0\\
0&0&0&0&0\\
0&0&0&0&0
\end{bmatrix}
$$

---

###### Input entry 2 at top-right

Because stride = 2, it is placed 2 columns to the right.

Contribution:

$$
\begin{bmatrix}
0&0&2&2&2\\
0&0&2&2&2\\
0&0&2&2&2\\
0&0&0&0&0\\
0&0&0&0&0
\end{bmatrix}
$$

---

###### Input entry 3 at bottom-left

Placed 2 rows down.

Contribution:

$$
\begin{bmatrix}
0&0&0&0&0\\
0&0&0&0&0\\
3&3&3&0&0\\
3&3&3&0&0\\
3&3&3&0&0
\end{bmatrix}
$$

---

###### Input entry 4 at bottom-right

Placed 2 rows down and 2 columns right.

Contribution:

$$
\begin{bmatrix}
0&0&0&0&0\\
0&0&0&0&0\\
0&0&4&4&4\\
0&0&4&4&4\\
0&0&4&4&4
\end{bmatrix}
$$

---

##### 7. Add all contributions

Now add the four matrices entrywise:

$$
\begin{bmatrix}
1&1&3&2&2\\
1&1&3&2&2\\
4&4&10&6&6\\
3&3&7&4&4\\
3&3&7&4&4
\end{bmatrix}
$$

That is exactly the output.